In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder,OrdinalEncoder,StandardScaler

In [4]:
data = pd.read_csv("../data/student_mental_health_clean.csv")

print(data.head())
print("---------------les dimensions----------------")
print(data.shape)


  Student_ID  Age  Gender  Year_of_Study  Sleep_Hours  Study_Hours_Per_Week  \
0    STU0001   22    Male              3          6.6                  17.6   
1    STU0002   21    Male              4          5.6                  22.4   
2    STU0003   22    Male              5          7.1                  28.1   
3    STU0004   24    Male              1          4.5                  14.8   
4    STU0005   20  Female              1          6.7                  17.2   

    GPA  Social_Media_Hours  Screen_Time_Hours Physical_Activity  \
0  2.91                 6.9                7.5          Moderate   
1  2.92                 7.6               11.5               Low   
2  2.68                 6.5                8.9               Low   
3  2.23                 5.5                7.6              High   
4  2.69                 4.3                7.0          Moderate   

   Financial_Stress  Family_Support  Attendance_Rate Part_Time_Job  \
0               8.5             4.0           

In [5]:
X = data.drop(columns=["Student_ID","Mental_Health_Risk"])
y=data["Mental_Health_Risk"]

In [6]:
X_train,X_temp,y_train,y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)
X_val,X_test,y_val,y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

In [7]:
categorical_features = X.select_dtypes(include = ["object","str"]).columns.to_list()
numerical_features = X.select_dtypes(include = ["number"]).columns.to_list()

ordinal_features = ["Physical_Activity","Diet_Quality"]
one_hot_features = ["Gender"]
binary_features = ["Part_Time_Job","Previous_Mental_Health_History"]


print("numerique :", numerical_features)
print("ordinales :", ordinal_features)
print("nominales :", one_hot_features)
print("binaires :", binary_features)


numerique : ['Age', 'Year_of_Study', 'Sleep_Hours', 'Study_Hours_Per_Week', 'GPA', 'Social_Media_Hours', 'Screen_Time_Hours', 'Financial_Stress', 'Family_Support', 'Attendance_Rate']
ordinales : ['Physical_Activity', 'Diet_Quality']
nominales : ['Gender']
binaires : ['Part_Time_Job', 'Previous_Mental_Health_History']


In [8]:
print(X["Physical_Activity"].unique())
print(X["Diet_Quality"].unique())
print(X["Gender"].unique())

<ArrowStringArray>
['Moderate', 'Low', 'High']
Length: 3, dtype: str
<ArrowStringArray>
['Poor', 'Average', nan, 'Good']
Length: 4, dtype: str
<ArrowStringArray>
['Male', 'Female', 'Non-binary']
Length: 3, dtype: str


In [9]:
num_pipline = Pipeline([
    ("imputer",SimpleImputer(strategy="median")),
    ("scaler",StandardScaler())
])

ordinal_pipline = Pipeline([
    ("imputer",SimpleImputer(strategy="most_frequent")),
    ("encoder",OrdinalEncoder(
        categories=[
            ["Low", "Moderate", "High"],
            ["Poor", "Average", "Good"]
        ],
        handle_unknown="use_encoded_value",
        unknown_value=-1
    ))
])

onehot_pipline = Pipeline([
    ("imputer",SimpleImputer(strategy="most_frequent")),
    ("encoder",OneHotEncoder(handle_unknown="ignore"))
])

binary_pipline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder",OneHotEncoder(drop="if_binary",handle_unknown="ignore"))
])

preprocessing = ColumnTransformer([
    ("num",num_pipline,numerical_features),
    ("ordinal",ordinal_pipline,ordinal_features),
    ("onehot",onehot_pipline,one_hot_features),
    ("binary",binary_pipline,binary_features),
])

In [10]:
X_train_processed = preprocessing.fit_transform(X_train)

X_val_processed = preprocessing.transform(X_val)
X_test_processed = preprocessing.transform(X_test)

print("Train :",X_train_processed.shape)
print("Validation :",X_val_processed.shape)
print("test :",X_test_processed.shape)



Train : (1400, 17)
Validation : (300, 17)
test : (300, 17)


In [11]:
# virification
print("les valeur manquante :",np.isnan(X_train_processed).sum())
print("les valeur manquante :",np.isnan(X_val_processed).sum())
print("les valeur manquante :",np.isnan(X_test_processed).sum())

les valeur manquante : 0
les valeur manquante : 0
les valeur manquante : 0


In [12]:
# les données sont presque équilibrées
print(y_train.value_counts())

Mental_Health_Risk
High      476
Low       462
Medium    462
Name: count, dtype: int64


In [13]:
# save preprocessing

import joblib
joblib.dump(preprocessing,"../models/preprocessing.joblib")

['../models/preprocessing.joblib']